In [26]:
import numpy as np

In [27]:
P = 768
J = 3
L = 12
l_h = 6

In [28]:
a_vec = [763, 679, 397, 61, 697, 373, 289, 257, 625, 41, 193, 449]
b_vec = [435, 69, 330, 18, 612, 246, 496, 640, 200, 524, 672, 672]

In [29]:
def gen_g(a_vec):
    G = np.zeros((36, 12), dtype=int)
    for i in range(6):
        for j in range(6):
            G[6*i+j, i] = (1 - a_vec[6+j])
            G[6*i+j, 6+j] = (a_vec[i]-1)
    return G

def gen_g_ab(G):
    all_indices = [i for i in range(l_h**2)]
    gb_indices = [3, 8]
    ga_indices = [i for i in all_indices if i not in gb_indices]
    Ga = G[ga_indices]
    Gb = G[gb_indices]
    return Ga, Gb

In [30]:
G = gen_g(a_vec)
Ga, Gb = gen_g_ab(G)

In [31]:
import math
import numpy as np

def solve_snf(Ga_list, P):
    """
    スミス標準形を用いて、Ga * b ≡ 0 (mod P) を満たす解空間を完全に特定する。
    
    戻り値:
        y_ranges: 各パラメータ y_i の (値のステップ幅, 取り得る値の個数) のリスト
        V: パラメータベクトル y から元の解ベクトル b を復元するための変換行列
    """
    # NumPy配列が渡された場合でもエラーにならないように、len()で判定
    if len(Ga_list) == 0 or len(Ga_list[0]) == 0:
        return [], []

    n = len(Ga_list)
    m = len(Ga_list[0])
    
    # NumPy配列の場合は標準のリストに変換し、要素を整数として扱う
    if isinstance(Ga_list, np.ndarray):
        A = Ga_list.tolist()
    else:
        A = [list(row) for row in Ga_list]
        
    # 変換行列 U, V を単位行列で初期化
    U = [[1 if i == j else 0 for j in range(n)] for i in range(n)]
    V = [[1 if i == j else 0 for j in range(m)] for i in range(m)]

    # 行・列の基本変形関数群
    def row_add(i, j, q):
        for k in range(m): A[i][k] -= q * A[j][k]
        for k in range(n): U[i][k] -= q * U[j][k]
    def col_add(i, j, q):
        for k in range(n): A[k][i] -= q * A[k][j]
        for k in range(m): V[k][i] -= q * V[k][j]
    def row_swap(i, j):
        A[i], A[j] = A[j], A[i]
        U[i], U[j] = U[j], U[i]
    def col_swap(i, j):
        for k in range(n): A[k][i], A[k][j] = A[k][j], A[k][i]
        for k in range(m): V[k][i], V[k][j] = V[k][j], V[k][i]
    def col_mult(i, q):
        for k in range(n): A[k][i] *= q
        for k in range(m): V[k][i] *= q

    # スミス標準形の対角化プロセス
    for t in range(min(n, m)):
        while True:
            # 対象の小行列内で絶対値が最小の非ゼロ要素を探す
            min_val = float('inf')
            pi, pj = -1, -1
            for i in range(t, n):
                for j in range(t, m):
                    if A[i][j] != 0 and abs(A[i][j]) < min_val:
                        min_val = abs(A[i][j])
                        pi, pj = i, j
            
            if pi == -1:
                break # 残りの要素がすべてゼロなら終了
                
            if pi != t: row_swap(t, pi)
            if pj != t: col_swap(t, pj)
            if A[t][t] < 0: col_mult(t, -1)
            
            changed = False
            for j in range(t + 1, m):
                if A[t][j] != 0:
                    q = A[t][j] // A[t][t]
                    col_add(j, t, q)
                    changed = True
            for i in range(t + 1, n):
                if A[i][t] != 0:
                    q = A[i][t] // A[t][t]
                    row_add(i, t, q)
                    changed = True
            
            # ピボットが他の要素を割り切れない場合の処理
            if not changed:
                divides_all = True
                for i in range(t + 1, n):
                    for j in range(t + 1, m):
                        if A[i][j] % A[t][t] != 0:
                            row_add(t, i, -1)
                            changed = True
                            divides_all = False
                            break
                    if not divides_all: break
                if not changed:
                    break

    # 対角成分を正にする
    for i in range(min(n, m)):
        if A[i][i] < 0:
            col_mult(i, -1)

    # パラメータ y_i の取り得る範囲を計算
    y_ranges = []
    for i in range(m):
        d_i = A[i][i] if i < min(n, m) else 0
        
        if d_i == 0:
            # 0 * y_i ≡ 0 なので、y_i は 0 から P-1 のすべての値を取れる
            step = 1
            num_vals = P
        else:
            # d_i * y_i ≡ 0 (mod P) の解は、y_i が P / gcd(d_i, P) の倍数のとき
            g = math.gcd(d_i, P)
            step = P // g
            num_vals = g
            
        y_ranges.append((step, num_vals))
        
    return y_ranges, V

In [32]:
import sympy as sp

def solve_hnf(Ga_list, P):
    """
    エルミート標準形のアプローチを用いて、Ga * b ≡ 0 (mod P) を満たす解の基底を求める。
    """
    # NumPy配列が渡された場合のエラーを回避するため、len()で判定する
    if len(Ga_list) == 0 or len(Ga_list[0]) == 0:
        return []
        
    rows = len(Ga_list)
    cols = len(Ga_list[0])
    
    # SymPyのMatrixはNumPy配列をそのまま受け取ることができる
    Ga = sp.Matrix(Ga_list)
    
    # 単位行列の P 倍を作成
    P_I = P * sp.eye(rows)
    
    # ブロック行列 M = [Ga, P*I] を作成 (サイズ: rows x (cols + rows))
    M = Ga.row_join(P_I)
    
    # M の有理数カーネルを取得し、整数空間の基底に変換する
    null_basis_rat = M.nullspace()
    
    basis_vectors = []
    for vec in null_basis_rat:
        # ベクトルの要素の分母の最小公倍数(LCM)を掛けて整数ベクトルにする
        lcm_val = 1
        for val in vec:
            lcm_val = sp.lcm(lcm_val, val.q)
        
        int_vec = vec * lcm_val
        
        # 上位 cols 個の要素が、求めるベクトル b に対応する
        b_vec = [int(int_vec[i]) % P for i in range(cols)]
        
        # すべてが0のベクトルは基底から除外
        if any(v != 0 for v in b_vec):
            basis_vectors.append(b_vec)
            
    # 重複する基底ベクトルを削除
    unique_basis = []
    for b in basis_vectors:
        if b not in unique_basis:
            unique_basis.append(b)
            
    return unique_basis

In [33]:
def extract_snf_basis(y_ranges, V, P):
    """
    SNFの出力結果から、解空間を生成するための基底ベクトル群を抽出する。
    
    戻り値:
        basis_info: 辞書のリスト。各要素は以下のキーを持つ
            - 'vector': 基底ベクトル (長さ12のリスト)
            - 'num_vals': この基底ベクトルに掛けられる係数の上限 (0 から num_vals-1 まで)
    """
    basis_info = []
    cols = len(V)
    m = len(y_ranges)
    
    for j in range(m):
        step, num_vals = y_ranges[j]
        
        # num_vals が 1 の場合、そのパラメータは常に 0 (mod P) なので基底に寄与しない
        if num_vals <= 1:
            continue
            
        # V の j 列目に step を掛けたベクトルが、このパラメータに対応する基底ベクトルとなる
        basis_vec = [0] * cols
        for i in range(cols):
            basis_vec[i] = (V[i][j] * step) % P
            
        # すべてが0のベクトルは除外
        if any(val != 0 for val in basis_vec):
            # 基底ベクトルとその係数が取り得る範囲をセットにして保存
            basis_info.append({
                'vector': basis_vec,
                'num_vals': num_vals
            })
    return basis_info


In [34]:
ker_G_h = solve_hnf(G, P)
ker_Ga_h = solve_hnf(Ga, P)
ker_Gb_h = solve_hnf(Gb, P)

In [35]:
y_ranges_G, V_G  = solve_snf(G, P)
y_ranges_Ga, V_Ga = solve_snf(Ga, P)
y_ranges_Gb, V_Gb = solve_snf(Gb, P)

In [36]:
ker_G_s = extract_snf_basis(y_ranges_G, V_G, P)
ker_Ga_s = extract_snf_basis(y_ranges_Ga, V_Ga, P)
ker_Gb_s = extract_snf_basis(y_ranges_Gb, V_Gb, P)

In [37]:
import numpy as np
import itertools

def analyze_diff_space_fast(basis_info_Ga, Gb, P):
    """
    GbがNumPy配列であることを前提に、条件判定に影響する基底を分離し、
    差空間の有効なパラメータパターンを高速に抽出する。
    """
    free_bases = []          # 係数が完全に自由な基底
    constrained_bases = []   # 条件判定に影響する基底
    
    # 1. 各基底が条件判定に影響するかを分類する
    for info in basis_info_Ga:
        # 計算を高速化するために基底ベクトルをNumPy配列に変換する
        vec = np.array(info['vector']) 
        
        # W = Gb * vec (mod P) をNumPyの機能で一括計算
        W = np.dot(Gb, vec) % P
        
        # W のすべての要素が 0 なら、この基底は条件Bに一切影響しない
        if np.all(W == 0):
            free_bases.append(info)
        else:
            info['W'] = W  # 計算済みの W をNumPy配列として保存
            constrained_bases.append(info)
            
    print(f"完全に自由な基底の数: {len(free_bases)}")
    print(f"制約を受ける基底の数: {len(constrained_bases)}")
    
    valid_constrained_patterns = []
    
    if not constrained_bases:
        print("制約を受ける基底がありません。条件Bを満たす解は存在しません。")
        return free_bases, constrained_bases, valid_constrained_patterns
        
    # 制約を受ける基底の係数の取り得る範囲をリスト化
    c_ranges = [range(info['num_vals']) for info in constrained_bases]
    rows_gb = Gb.shape[0]
    
    # 2. 影響する基底の係数だけを全探索して有効な組み合わせを見つける
    for coeffs in itertools.product(*c_ranges):
        # ゼロベクトルで初期化
        current_W = np.zeros(rows_gb, dtype=int)
        
        for c, info in zip(coeffs, constrained_bases):
            # Numpyの配列演算で足し合わせる
            current_W = (current_W + c * info['W']) % P
            
        # 条件Bの判定: 結果の「すべての要素」が 0 ではないこと
        if np.all(current_W != 0):
            valid_constrained_patterns.append(coeffs)
            
    print(f"制約基底の有効な係数パターン数: {len(valid_constrained_patterns)}")
    
    return free_bases, constrained_bases, valid_constrained_patterns

In [38]:
import random

def generate_solution_from_patterns(free_bases, constrained_bases, valid_patterns, P, cols=12):
    """
    特定された自由基底と有効な制約パターンから、差空間の解を1つ高速に生成する。
    戻り値はNumPy配列のベクトルとなる。
    """
    if not valid_patterns:
        return None
        
    b = np.zeros(cols, dtype=int)
    
    # 1. 自由な基底には、0 から num_vals - 1 までのランダムな係数を掛けて足す
    for info in free_bases:
        c = random.randint(0, info['num_vals'] - 1)
        vec = np.array(info['vector'])
        b = (b + c * vec) % P
            
    # 2. 制約を受ける基底には、有効なパターンから1つ選んで掛けて足す
    pattern = random.choice(valid_patterns)
    for c, info in zip(pattern, constrained_bases):
        vec = np.array(info['vector'])
        b = (b + c * vec) % P
            
    return b


In [39]:
free, const, valid_pats = analyze_diff_space_fast(ker_Ga_s, Gb, P)

# 高速に解を5つ生成する
for _ in range(5):
    sol = generate_solution_from_patterns(free, const, valid_pats, P)
    print(sol.tolist())

完全に自由な基底の数: 11
制約を受ける基底の数: 1
制約基底の有効な係数パターン数: 48
[744, 240, 288, 672, 96, 96, 640, 128, 192, 64, 0, 512]
[381, 747, 726, 462, 60, 330, 272, 0, 696, 308, 352, 480]
[198, 642, 708, 276, 552, 60, 352, 640, 272, 440, 64, 64]
[420, 372, 744, 552, 144, 120, 704, 128, 352, 432, 384, 128]
[492, 588, 216, 600, 48, 72, 64, 512, 288, 464, 128, 512]


In [40]:
b_vec = [657, 279, 366, 102, 12, 210, 720, 0, 280, 388, 224, 736]

In [ ]:
display(Ga @ b_vec % P)
display(Gb @ b_vec % P)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

array([576, 384])

: 